In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import inception_v3, Inception_V3_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"

what is Inception-v3?

In [9]:
# Load pretrained Inception-v3
weights = Inception_V3_Weights.DEFAULT
inception = inception_v3(weights=weights, aux_logits=True).to(device)
inception.eval()

# Replace final classifier with identity so output is 2048-dim features
inception.fc = nn.Identity()

# Torchvision Inception expects 299x299 images and ImageNet normalization
preprocess = weights.transforms()

In [12]:
@torch.no_grad()
def extract_inception_features(images, batch_size=256):
    """
    images: torch.Tensor of shape [N, 3, H, W]

    Expected input range:
      - [0, 1] float images, or
      - [-1, 1] float images if you uncomment the conversion below

    returns:
      features of shape [N, 2048]
    """
    inception.eval()
    features = []

    for i in range(0, len(images), batch_size):
        x = images[i:i + batch_size].to(device)

        # If your generated images are in [-1, 1], uncomment this:
        # x = (x.clamp(-1, 1) + 1) / 2

        # Resize + ImageNet normalize
        x = preprocess(x)

        feat = inception(x)

        # In case torchvision returns an InceptionOutputs object
        if not isinstance(feat, torch.Tensor):
            feat = feat.logits

        features.append(feat.cpu())

    return torch.cat(features, dim=0)


# Example:
# real_images: [N, 3, 32, 32], float in [0, 1]
# fake_images: [N, 3, 32, 32], float in [0, 1]

N = 100

real_images = torch.rand(N, 3, 32, 32)
fake_images = torch.rand(N, 3, 32, 32)

real_features = extract_inception_features(real_images)
fake_features = extract_inception_features(fake_images)

print(real_features.shape)  # [N, 2048]
print(fake_features.shape)  # [N, 2048]

torch.Size([100, 2048])
torch.Size([100, 2048])


In [14]:
def ploynomial_kernel(x, y, degree=3):
    scale = 1.0 / x.shape[1]
    return (scale * x @ y.T + 1) ** degree

## KID:
- take N real and N fake images
- extract a k-dimensional (2048, say) feature vector for each image (using torchvision) $\quad\implies\phi$
- get pairwise kernel similarities $\quad\implies k(x_i, y_i)$
    - 3 sets: (real, real), (fake, fake) and (real, fake)
    - feed those 3 similarities through formula

In [27]:
def kid_estimate(real_features, fake_features):
    k_xx = ploynomial_kernel(real_features, real_features)
    k_yy = ploynomial_kernel(fake_features, fake_features)
    k_xy = ploynomial_kernel(real_features, fake_features)

    sum_k_xx = k_xx.sum() - k_xx.diag().sum()
    sum_k_yy = k_yy.sum() - k_yy.diag().sum()
    sum_k_xy = k_xy.sum()

    mmd = (sum_k_xx / (N * (N - 1))) + (sum_k_yy / (N * (N - 1))) - (2 * sum_k_xy / (N * N))
    return mmd.item()

kid_score = kid_estimate(real_features, fake_features)
print("KID Score:", kid_score)

KID Score: -0.0003790855407714844


In [36]:
def kid_score(real_features, fake_features, subsets=5, subset_size=2):
    device = real_features.device
    scores = []
    
    n_real = real_features.shape[0]
    n_fake = fake_features.shape[0]
    
    if n_real < subset_size or n_fake < subset_size:
        raise ValueError("subset_size must be no larger than both feature sets.")
    
    for _ in range(subsets):
        real_idx = torch.randperm(subset_size, device=device)
        fake_idx = torch.randperm(subset_size, device=device)
        
        x = real_features[real_idx]
        y = fake_features[fake_idx]
        
        kid_est = kid_estimate(x, y)
        # print(f"KID estimate for subset: {kid_est}")
        scores.append(kid_est)
        # print(scores)
    
    # print(type(scores))
    scores = torch.tensor(scores)
    # print(scores.shape)
    return scores.mean().item(), scores.std().item()

In [37]:
kid_score(real_features, fake_features)

(-0.0006260976078920066, 1.0493535207434235e-10)